# Prep for website
Form the same as the kelp outputs
* Split out kelp geometries
* Array of named locations and info for satellite images

In [1]:
import geopandas
import leafmap
import pandas
import pathlib
import dask.distributed
import datetime

module_path = pathlib.Path.cwd().parent / 'scripts'
import sys
if str(module_path) not in sys.path:
    sys.path.append(str(module_path))
import utils
import sentinel2
import training
import sampling

%load_ext autoreload
%autoreload 2

In [2]:
cluster = dask.distributed.LocalCluster()
client = dask.distributed.Client(cluster)
display(client)

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:62080/status,
Dashboard: http://127.0.0.1:62080/status,Workers: 4
Total threads: 8,Total memory: 31.73 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:62081,Workers: 0
Dashboard: http://127.0.0.1:62080/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:62101,Total threads: 2
Dashboard: http://127.0.0.1:62102/status,Memory: 7.93 GiB
Nanny: tcp://127.0.0.1:62084,


# Value to edit

In [3]:
all_training_sites =  ["CatlinsLake", "CatlinsRiverMouth", "Childrens", "Duvauchelle", "Robinsons", "Takamatua",
                       "Purau", "Ihutai", "IveyBay_Nov25", "IveyBay_Feb26", "LeftBank_Nov25", "LeftBank_Feb26", "Paremata_Nov25", "Paremata_Feb26",
              "Paremata_Feb25", "ThePoint_Nov25", "ThePoint_Feb26", "Takapuwahia_Nov25", "Takapuwahia_Feb26", "IveyBay_ThePoint_LeftBank_Oct24"]
large_seagrass_sites =  ["CatlinsRiverMouth", "Childrens", "Duvauchelle", "Robinsons", "Ihutai", "LeftBank_Nov25", "ThePoint_Nov25"]

In [4]:
satellite_classes_all = {'Seagrass': 1, 'Seagrass submerged': 2, 'Gracilaria': 3, 'Gracilaria submerged': 4, 
                           'Ulva': 5, 'Submerged vegetation': 9, 'Microphytobenthos': 10, 'Satmarsh': 14,
                           'Unvegetated': 15, 'Water': 16, 'Terrestrial': 18, 'Rock': 19, 'Mixed': 22,}
satellite_classes_all_unet = {'Seagrass': 0, 'Seagrass submerged': 1, 'Gracilaria': 2, 'Gracilaria submerged': 3, 
                           'Ulva': 4, 'Submerged vegetation': 5, 'Microphytobenthos': 6, 'Satmarsh': 7,
                           'Unvegetated': 8, 'Water': 9, 'Terrestrial': 10, 'Rock': 11, 'Mixed': 12,}

In [29]:
sample_method = "sampling_2"
method_2_threshold = .50
model_max_cloud_cover = 0 # percentage
model_low_tide_delta_hrs = 0
model_low_tide_delta_mins = 30
test_threshold = 0.1

predict_max_cloud_cover = 1 # percentage
predict_low_tide_delta_hrs = 0
predict_low_tide_delta_mins = 30

model_name = f"test_on_{int(test_threshold*100)}_percent_all_classes.joblib"

website_tab = (
    f"RF_model_{int(test_threshold*100)}_percent_test_{utils.get_samples_folder(sample_method, method_2_threshold)}_"
    f"{utils.get_low_tide_max_cloud_name(model_low_tide_delta_hrs, model_low_tide_delta_mins, model_max_cloud_cover)}_"
    f"predict_over_{utils.get_low_tide_max_cloud_name(predict_low_tide_delta_hrs, predict_low_tide_delta_mins, predict_max_cloud_cover)}"
)

In [30]:
model_path = utils.get_models_path(sample_method=sample_method, method_2_threshold=method_2_threshold,
                                   low_tide_delta_hrs=model_low_tide_delta_hrs, low_tide_delta_mins=model_low_tide_delta_mins,
                                   max_cloud_cover=model_max_cloud_cover)
model_file = model_path / model_name
prediction_sites = large_seagrass_sites
years = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

In [31]:
predictions_path, satellite_path = utils.get_prediction_path(sample_method, method_2_threshold, model_max_cloud_cover=model_max_cloud_cover, predict_max_cloud_cover=predict_max_cloud_cover,
                                             predict_low_tide_delta_hrs=predict_low_tide_delta_hrs, predict_low_tide_delta_mins=predict_low_tide_delta_mins,
                                             model_low_tide_delta_hrs=model_low_tide_delta_hrs, model_low_tide_delta_mins=model_low_tide_delta_mins)

# Cells to run
* Loop over prediction sites and years
* Split out seagrass extents for each date
* Save the identifying satellite information for each date

In [32]:
data_path = utils.get_data_path()
utils.create_data_folders()

In [33]:
website_options = {"value": prediction_sites, "label": prediction_sites}
website_options = pandas.DataFrame(website_options)
(data_path / "website" / website_tab).mkdir(exist_ok=True, parents=True)
website_options.to_csv(data_path / "website" / website_tab / "options.csv", index=False)

In [ ]:
for prediction_site in prediction_sites:
    output_folder = data_path / "website" / website_tab / prediction_site
    output_folder.mkdir(exist_ok=True, parents=True)

    if (output_folder / "seagrass_presence_absence_map.gpkg").exists() and False:
        print(f"{prediction_site} already run. Go to next")
        continue
    extents_all_years = {"Seagrass": [], "Ulva": [], "Gracilaria": []}
    info = {"ids": [], "percentages_2":[], "percentage_98":[], "date":[]}
    for year in years:
        print(f"{prediction_site}: year {year}")
        satellite_info_set = False
        for target in extents_all_years.keys():
            target_file = predictions_path / f"{prediction_site}_{year}_model_{model_file.stem}_{target}.gpkg"

            if not target_file.exists():
                print(f"WARNING no {target_file.name} file. Run 'predict_for_site.ipynb' is this is unexpected.")
                continue
            target_extents = geopandas.read_file(target_file)
            extents_all_years[target].append(target_extents)

            for index, row in target_extents.iterrows():
                target_extent = geopandas.GeoDataFrame(geometry=[row["geometry"]], crs=utils.CRS_NZTM)
                date_YYMMDD = datetime.datetime.strptime(row.date, '%Y-%m-%d %H:%M:%S.%f').strftime('%Y-%m-%d')
        
                filename = f"{date_YYMMDD}_{target.lower()}.gpkg"
                target_extent.to_file(output_folder / filename)

                # Look up the satellite tile information
                if not satellite_info_set:
                    #print(f"\tGet Tile(s) ID: {date_YYMMDD}")
                    site_polygon = geopandas.read_file(utils.get_site_polygon_path(prediction_site))
                    tile_id, percentage_2, percentage_98 = sentinel2.get_satellite_info(geometry=site_polygon, date_YYMMDD=date_YYMMDD)
                    
                    info["date"].append(row.date)
                    info["ids"].append(tile_id)
                    info["percentages_2"].append(percentage_2)
                    info["percentage_98"].append(percentage_98); 
            satellite_info_set = True
            
    info = pandas.DataFrame(info)
    info = info.rename(columns={"ids": "Satellite Tile IDs", "percentages_2": "Percentile 2", "percentage_98": "Percentile 98"})
    
    for key, value in extents_all_years.items():
        print(f"\tTarget {key}")
        target_extents = pandas.concat(value, ignore_index=True)
        target_extents['area'] = target_extents.area
        info[f"{key} Area [m^2]"] = target_extents.area
        target_extents.dissolve()[["geometry"]].to_file(output_folder / f"{key.lower()}_presence_absence_map.gpkg")

    
    info.to_csv(output_folder / "info_all_dates.csv", index=False)
    


CatlinsRiverMouth: year 2016
CatlinsRiverMouth: year 2017
WARNING no CatlinsRiverMouth_2017_model_test_on_10_percent_all_classes_Seagrass.gpkg file. Run 'predict_for_site.ipynb' is this is unexpected.
WARNING no CatlinsRiverMouth_2017_model_test_on_10_percent_all_classes_Ulva.gpkg file. Run 'predict_for_site.ipynb' is this is unexpected.
WARNING no CatlinsRiverMouth_2017_model_test_on_10_percent_all_classes_Gracilaria.gpkg file. Run 'predict_for_site.ipynb' is this is unexpected.
CatlinsRiverMouth: year 2018
CatlinsRiverMouth: year 2019
CatlinsRiverMouth: year 2020
CatlinsRiverMouth: year 2021
WARNING no CatlinsRiverMouth_2021_model_test_on_10_percent_all_classes_Seagrass.gpkg file. Run 'predict_for_site.ipynb' is this is unexpected.
WARNING no CatlinsRiverMouth_2021_model_test_on_10_percent_all_classes_Ulva.gpkg file. Run 'predict_for_site.ipynb' is this is unexpected.
WARNING no CatlinsRiverMouth_2021_model_test_on_10_percent_all_classes_Gracilaria.gpkg file. Run 'predict_for_site.ip